# **TaniMol: 07 - Export Results**

This notebook exports all computed results from the pipeline into portable, reusable files. The goal is to produce a self-contained `results/` directory that can be shared independently of the codebase — anyone receiving it gets the full picture without needing to rerun the pipeline.

**Exported files:**
- `cluster_statistics.csv` — per-cluster pIC50 summary (mean, median, std, min, max)
- `activity_cliffs.csv` — all detected cliff pairs with SMILES, similarity, ΔpIC50, SALI
- `molecule_summary.csv` — per-molecule table with cluster assignment, max SALI, cliff involvement
- `pipeline_summary.json` — run metadata, parameters, and key results
- `figures/` — all plots as PNG (300 DPI, print quality)

### **1. Load Data and Recompute Results**

Load the clustering results and preprocessed dataset, then recompute all activity analysis outputs. This ensures the exports reflect the exact same state as the analysis notebooks.

In [1]:
from src.config import PROCESSED_DIR
from src.activity_analysis import (
    within_cluster_activity_distributions, activity_cliffs, sali,
    similarity_activity_correlation,
)
from src.export import (
    export_cluster_statistics, export_activity_cliffs,
    export_molecule_summary, export_pipeline_summary, export_all_figures,
)
import pandas as pd
import numpy as np
import pickle

with open(f"{PROCESSED_DIR}/clustering_results.pkl", "rb") as f:
    results = pickle.load(f)

morgan_sim = results["morgan"]["similarity_matrix"]
morgan_clusters = results["morgan"]["clusters"]
morgan_singletons = results["morgan"]["singletons"]

df = pd.read_csv(f"{PROCESSED_DIR}/cleaned_activities.csv")
pic50 = df["pchembl_value"].values
delta_pic50 = np.abs(pic50[:, None] - pic50[None, :])

cluster_stats = within_cluster_activity_distributions(morgan_clusters, pic50)
cliffs = activity_cliffs(morgan_sim, delta_pic50)
sali_matrix, sali_values = sali(morgan_sim, delta_pic50)
rho, p_value = similarity_activity_correlation(morgan_sim, delta_pic50)

### **2. Export Data**

Write all tabular results and metadata to `results/`. Each file is self-contained and can be opened in Excel, pandas, or any JSON viewer without additional context.

In [2]:
export_cluster_statistics(cluster_stats)
export_activity_cliffs(cliffs, df, sali_matrix)
export_molecule_summary(df, morgan_clusters, morgan_singletons, sali_matrix, cliffs)
export_pipeline_summary(df, morgan_clusters, morgan_singletons, cliffs, sali_values, rho, p_value)

Exported 339 cluster statistics to /home/stanuch/Dev/TaniMol/results/cluster_statistics.csv
Exported 37 activity cliffs to /home/stanuch/Dev/TaniMol/results/activity_cliffs.csv
Exported 3324 molecule summaries to /home/stanuch/Dev/TaniMol/results/molecule_summary.csv
Exported pipeline summary to /home/stanuch/Dev/TaniMol/results/pipeline_summary.json


{'generated_at': '2026-04-16T22:50:09.185597',
 'chembl_version': '36',
 'targets': {'CHEMBL3105': 'PARP1'},
 'parameters': {'fingerprint_type': 'Morgan (ECFP4)',
  'morgan_radius': 2,
  'clustering_threshold': 0.6,
  'activity_types': ['IC50'],
  'min_confidence_score': 7,
  'cliff_sim_threshold': 0.8,
  'cliff_activity_threshold': 2.0},
 'dataset': {'n_molecules': 3324,
  'pic50_mean': 7.128,
  'pic50_median': 7.3,
  'pic50_std': 1.15,
  'pic50_range': [4.0, 10.7]},
 'clustering': {'n_clusters': 339,
  'n_singletons': 370,
  'n_clustered_molecules': 2954,
  'largest_cluster_size': 173},
 'activity_cliffs': {'n_cliffs': 37, 'max_delta_pic50': 4.05},
 'sali': {'max': 72.8,
  'mean': 1.57,
  'median': 1.33,
  'p95': 3.79,
  'p99': 4.8,
  'pairs_above_50': 4},
 'spearman_correlation': {'rho': -0.1149,
  'p_value': 0.0,
  'sar_confirmed': True}}

### **3. Export Figures**

Save all activity analysis plots to `results/figures/` as high-resolution PNGs. These are publication-ready — white background, 300 DPI, tight bounding boxes.

In [3]:
from src.visualization import (
    plot_cluster_activity_boxplots, plot_activity_cliff_scatter,
    plot_sali_distribution, plot_similarity_activity_density,
)

export_all_figures({
    "cluster_activity_boxplots": lambda: plot_cluster_activity_boxplots(morgan_clusters, pic50),
    "activity_cliff_scatter": lambda: plot_activity_cliff_scatter(morgan_sim, delta_pic50),
    "sali_distribution": lambda: plot_sali_distribution(sali_values),
    "similarity_activity_density": lambda: plot_similarity_activity_density(morgan_sim, delta_pic50, rho, p_value),
})

Exported 4 figures to /home/stanuch/Dev/TaniMol/results/figures
